<h1>Librerías</h1>

In [115]:
import numpy as np
import obspy
import emd
import pandas as pd
from tqdm.notebook import tqdm
import os
import scipy.signal as sg
from concurrent.futures import ThreadPoolExecutor
%matplotlib inline
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from IPython.display import clear_output

import librosa
import pywt
from scipy.stats import entropy
import nolds

<h1>Funciones básicas</h1>

In [116]:
# Función de filtro pasa-banda utilizando el diseño de filtro Butterworth.
# Filtra los datos entre las frecuencias lowc y high.
def butter_bandpass_filter(senal: np.array, lowcut: float, highcut: float, fs: float, order: int):
    nyquist = 0.5 * fs  # Frecuencia de Nyquist, la mitad de la tasa de muestreo
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = sg.butter(order, [low, high], btype='band', analog=False)  # Diseña el filtro pasa-banda Butterworth
    y = sg.filtfilt(b, a, senal)  # Aplica el filtro a los datos usando filtrado cero-fase
    return y

In [117]:
def notch_filter(senal, sr, freq, quality):

    # Diseñar el filtro notch
    b, a = sg.iirnotch(freq, quality, sr)

    # Aplicar el filtro a la señal
    senal_filtrada = sg.lfilter(b, a, senal)

    return senal_filtrada

In [118]:
# Extraer MFCCs 

def extraer_caracteristicas_mfccs(audio, frecuencia, numero):
    mfccs = librosa.feature.mfcc(y=audio, sr=frecuencia, n_mfcc=numero)  #n:mfcc a evaluar según rendimiento del modelo

    mfccs_mean = np.mean(mfccs, axis=1)
    mfccs_std = np.std(mfccs, axis=1)
    mfccs_max = np.max(mfccs, axis=1)
    mfccs_min = np.min(mfccs, axis=1)


    # Concatenar estadísticas de MFCCs
    caracteristicas_mfcc = np.concatenate([mfccs_mean, mfccs_std, mfccs_max, mfccs_min])
    return caracteristicas_mfcc



In [119]:
# Función para extraer características wavelet
def extraer_caracteristicas_wavelet(senal, wavelet='db4', nivel=5):
    coeficientes = pywt.wavedec(senal, wavelet, level=nivel)
    caracteristicas_wavelet = []
    for coef in coeficientes:
        caracteristicas_wavelet.append(np.mean(coef))  # Media
        caracteristicas_wavelet.append(np.std(coef))   # Desviación estándar
        caracteristicas_wavelet.append(np.max(coef))   # Máximo
        caracteristicas_wavelet.append(np.min(coef))   # Mínimo
    return np.array(caracteristicas_wavelet)


In [120]:
def calcular_entropia(senal):
    hist, _ = np.histogram(senal, bins=50, density=True)
    hist = hist / np.sum(hist)
    return entropy(hist)

In [129]:
def extract_respiratory_features(audio, sr):
    """Extrae todas las características relevantes para respiración."""
    
    mfcc_features = extraer_caracteristicas_mfccs(audio, sr, 10)
    
    # 3. Características wavelet
    wavelet_features = extraer_caracteristicas_wavelet(audio)

    # Calcular características de complejidad
    entropia_shannon = calcular_entropia(audio)
    dimension_fractal = nolds.hurst_rs(audio)
    variabilidad = np.std(np.diff(audio))

    # Combinar todas las características
    caracteristicas = np.concatenate([mfcc_features, wavelet_features, [entropia_shannon, dimension_fractal, variabilidad]])

    # Combinar todas las características
    return caracteristicas

In [122]:
# Ruta de la carpeta donde están los archivos de audio
folder_path = "./Respiratory_Sound_Database\Respiratory_Sound_Database//audio_and_txt_files"
# Obtener lista de archivos de audio en la carpeta
audio_files = [f for f in os.listdir(folder_path) if f.endswith(".wav")]


patients = pd.read_csv("Respiratory_Sound_Database\Respiratory_Sound_Database\patient_diagnosis.csv",  header=None)

In [123]:
def process_file(idx):
    # 1. Validación de archivo
    file_name = audio_files[idx]
    audio_path = os.path.join(folder_path, file_name)
    
    if not os.path.exists(audio_path):
        print(f"Archivo no encontrado: {audio_path}")
        return None

    
    try:
        # 2. Cargar con SR original y re-muestrear
        audio, orig_sr = librosa.load(audio_path, sr=None)  # Carga con SR nativo
        audio, sr = resample_audio(audio, orig_sr, 22050)
        
        # 3. Validar duración después de re-muestreo
        if len(audio) < 0.5 * sr:  # Mínimo 0.5 segundos
            print(f"Audio demasiado corto: {file_name}")
            
            return None

        # 4
        patient_id = int(file_name[:3])
        sick = patients.loc[patients[0] == patient_id, 1].values[0]
    except Exception as e:
        print(f"Error cargando {file_name}: {str(e)}")
        return None
    
    # 5. Preprocesamiento de señal
    audio = butter_bandpass_filter(audio, 100, 1800, sr, order=4)
    audio = notch_filter(audio, sr, freq=50, quality=35)
    
    # 6. Segmentación en ventanas
    windows = segment_audio(audio, sr, window_sec=10, hop_sec=0.5)
    print(windows)
    # 7. Extracción de características por ventana
    all_features = []
    for window in windows:
        print(len(window), sr)

        if len(window) < 0.5 * sr:  # Ignorar ventanas muy cortas
            
            continue
        features = extract_respiratory_features(window, sr)
        all_features.append(features)
    
    
    if not all_features:
        print(f"Audio demasiado corto: {file_name}")
        return None 
    
    # 6. Normalización y promediado
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(all_features)
    avg_features = np.mean(scaled_features, axis=0)
    
    return (avg_features, sick)


In [127]:
def process_file(idx):
    # 1. Validación de archivo
    file_name = audio_files[idx]
    audio_path = os.path.join(folder_path, file_name)
    
    if not os.path.exists(audio_path):
        print(f"Archivo no encontrado: {audio_path}")
        return None

    try:
        # 2. Cargar con SR original y re-muestrear
        audio, sr = librosa.load(audio_path, sr=None)
        #audio, sr = resample_audio(audio, orig_sr, 22050)
        
        # 3. Validar duración después de re-muestreo
        if len(audio) < 0.5 * sr:  # Mínimo 0.5 segundos
            print(f"Audio demasiado corto: {file_name}")
            return None

        # 4. Obtener etiqueta (sick)
        patient_id = int(file_name[:3])
        sick = patients.loc[patients[0] == patient_id, 1].values[0]
        
    except Exception as e:
        print(f"Error cargando {file_name}: {str(e)}")
        return None
    
    # 5. Preprocesamiento de señal
    audio = butter_bandpass_filter(audio, 100, 1800, sr, order=4)
    audio = notch_filter(audio, sr, freq=50, quality=35)
    
    # 6. Extracción de características para TODO el audio
    features = extract_respiratory_features(audio, sr)

    
    return (features, sick)


In [130]:
uno, dos = process_file(1)

In [131]:
uno

array([-7.42958175e+02,  1.59111096e+02,  7.49610581e+01,  5.84029015e+00,
       -1.18128814e+01,  8.26551386e+00,  2.70693770e+01,  2.28013932e+01,
        5.50865333e+00, -4.02898064e+00,  1.80490135e+01,  2.03181930e+01,
        1.19897124e+01,  1.03037541e+01,  7.80623172e+00,  5.70939203e+00,
        6.49981068e+00,  5.19772690e+00,  6.12973231e+00,  6.85900989e+00,
       -6.24426537e+02,  2.90657082e+02,  1.28430170e+02,  3.67371445e+01,
        1.11440981e+01,  2.89734135e+01,  4.63316607e+01,  3.84920998e+01,
        2.08695690e+01,  1.29531887e+01, -7.74855923e+02,  1.24973968e+02,
        2.67160390e+01, -2.79561052e+01, -5.33053865e+01, -1.35221862e+01,
        3.49576019e+00,  3.18240154e+00, -2.58959751e+01, -3.65442135e+01,
        1.33815772e-06,  1.01745803e-02,  1.77997358e-01, -2.18545273e-01,
        6.96316051e-06,  1.16929305e-03,  3.19182988e-02, -5.52572360e-02,
        1.43437779e-08,  3.51219454e-04,  1.66704230e-02, -1.31820193e-02,
       -6.62368633e-10,  

In [132]:
dos

'URTI'